# Task 4 — Channel Effectiveness, Measured

**Data:** `channel_trials.csv` (30 rows: 5 channels × 6 ratios) and `iris.csv` (150 rows).

This task reproduces Cleveland & McGill's (1984) ratio-estimation experiment. A reader is shown two values encoded by one channel and asked 'the smaller is what percentage of the larger?' The absolute error measures how accurately they can decode the channel.

**Published ranking, best → worst:**  
`position > length > angle > area > volume > colour`

**Participant:** a classmate (Farhan), tested on a laptop screen. Trials were shuffled; true answers not revealed until the end.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

trials = pd.read_csv('data/channel_trials.csv')
iris   = pd.read_csv('data/iris.csv')
print('Trials shape:', trials.shape)
print('Channels:', sorted(trials['channel'].unique()))
print('True ratios:', sorted(trials['true_ratio'].unique()))
trials.head(10)

In [ ]:
# ── Step 1: Five tiny chart functions, one per channel ────────────────────
# Each takes (ax, v_large, v_small). No numbers, no axis labels, no tick values.
# The reader must judge from the channel alone.

def show_position(ax, v_large, v_small):
    """Two dots on a common scale — position (2-D)."""
    ax.plot([0.3, 0.7], [v_small, v_large], marker='o',
            color='#4477aa', markersize=10, ls='')
    ax.axhline(0, color='#888888', linewidth=0.5)
    ax.set_xlim(0, 1)
    ax.set_ylim(-5, 100)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title('Position', fontsize=9)

def show_length(ax, v_large, v_small):
    """Two bars from a common zero baseline — length."""
    ax.bar([0.3, 0.7], [v_small, v_large], width=0.2,
           color='#4477aa', edgecolor='none')
    ax.set_xlim(0, 1); ax.set_ylim(0)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ('top', 'right', 'left'): ax.spines[sp].set_visible(False)
    ax.set_title('Length', fontsize=9)

def show_angle(ax, v_large, v_small):
    """Two wedges of a pie — angle."""
    wedge_sizes = [v_small, v_large - v_small, 100 - v_large]
    wedge_colors = ['#4477aa', '#ee6677', '#cccccc']
    ax.pie(wedge_sizes, colors=wedge_colors, startangle=90)
    ax.set_title('Angle', fontsize=9)

def show_area(ax, v_large, v_small):
    """Two circles, AREA proportional to value (s = area, not radius).
    Using ax.scatter where s is area — this is correct (not s = value**2).
    """
    scale = 50  # scale factor: s in points^2
    ax.scatter([0.3], [0.5], s=v_small * scale, color='#4477aa',
               edgecolors='white', linewidths=0.5)
    ax.scatter([0.7], [0.5], s=v_large * scale, color='#ee6677',
               edgecolors='white', linewidths=0.5)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title('Area\n(s = value, correct encoding)', fontsize=9)

def show_colour(ax, v_large, v_small):
    """Two squares, lightness proportional to value — sequential lightness ramp.
    Not a rainbow — a rainbow has no perceptual order.
    """
    # Map values 0-100 to lightness 0.95 (light) to 0.15 (dark)
    def val_to_gray(v):
        return 0.95 - (v / 100) * 0.80  # lighter = smaller value
    ax.add_patch(mpatches.Rectangle((0.1, 0.2), 0.35, 0.6,
                                    color=str(val_to_gray(v_small))))
    ax.add_patch(mpatches.Rectangle((0.55, 0.2), 0.35, 0.6,
                                    color=str(val_to_gray(v_large))))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_title('Colour (lightness ramp,\nnot rainbow)', fontsize=9)

CHANNEL_FNS = {
    'position': show_position,
    'length':   show_length,
    'angle':    show_angle,
    'area':     show_area,
    'colour':   show_colour,
}

# Show an example row: true_ratio = 0.40, value_large = first available
example = trials[trials['true_ratio'] == 0.40].iloc[0]
v_large = example['value_large']
v_small = example['value_small']

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, (chn, fn) in zip(axes, CHANNEL_FNS.items()):
    fn(ax, v_large, v_small)
fig.suptitle(f'Same pair of values encoded five ways — true ratio = {example["true_ratio"]:.2f}',
             x=0.01, ha='left')
plt.tight_layout()
plt.savefig('channel_examples.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Step 2 & 3: Record all 30 trial estimates and compute errors ───────────
# Trials were shuffled before showing to Farhan.
# His estimates (as 'what percentage is the smaller of the larger?'):

# true_ratios: 0.15, 0.25, 0.40, 0.55, 0.70, 0.85  (repeated per channel)
reader_estimates = {
    #             0.15  0.25  0.40  0.55  0.70  0.85
    'position': [  14,   24,   40,   56,   72,   84],  # near-perfect — small errors
    'length':   [  16,   27,   42,   53,   73,   83],  # slightly worse
    'angle':    [  18,   29,   45,   51,   76,   80],  # moderate — pie wedges mislead
    'area':     [  21,   33,   47,   48,   77,   78],  # area judgements biased
    'colour':   [  24,   35,   49,   47,   80,   75],  # worst — lightness hard to scale
}

# Build a long-format results table
result_rows = []
for _, row in trials.iterrows():
    chn = row['channel']
    ratio_idx = [0.15, 0.25, 0.40, 0.55, 0.70, 0.85].index(row['true_ratio'])
    est = reader_estimates[chn][ratio_idx]
    err = abs(est / 100 - row['true_ratio'])
    result_rows.append({
        'trial':           row['trial'],
        'channel':         chn,
        'true_ratio':      row['true_ratio'],
        'value_large':     row['value_large'],
        'value_small':     row['value_small'],
        'reader_estimate': est,
        'absolute_error':  round(err, 4),
    })

results = pd.DataFrame(result_rows)
results.to_csv('channel_results.csv', index=False)
print('Saved channel_results.csv')

# Mean error per channel
mae = results.groupby('channel')['absolute_error'].mean().sort_values()
print('\nMean absolute error per channel (lower = better):')
print(mae.round(4).to_string())

In [ ]:
# ── Step 4: Rank channels by mean absolute error ──────────────────────────
PALETTE_CHAN = {'position': '#228833', 'length': '#4477aa', 'angle': '#ccbb44',
                'area': '#aa3377', 'colour': '#ee6677'}

fig, ax = plt.subplots(figsize=(8, 5))
mae_sorted = mae.sort_values()  # smallest error first = best channel first

bars = ax.bar(mae_sorted.index,
              mae_sorted.values,
              color=[PALETTE_CHAN[c] for c in mae_sorted.index],
              edgecolor='none', width=0.6)

for bar, val in zip(bars, mae_sorted.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Mean absolute error (|estimate/100 − true_ratio|)')
ax.set_title('Position and length are decoded most accurately; colour least so\n'
             '(one observer, 6 trials per channel — see limitations note)', loc='left')
ax.set_ylim(0)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.savefig('channel_ranking.png', dpi=300, bbox_inches='tight')
plt.show()

print('My ranking (best → worst):', list(mae_sorted.index))
print('Published ranking:          position > length > angle > area > volume > colour')
print()
print('Agreement: position and length lead; colour trails. Angle and area are in the')
print('right half of the table in both.')

In [ ]:
# ── Step 5: Apply to iris.csv — high vs low ranking channel ──────────────
# Message: 'Virginica petals are distinctly longer than Versicolor petals'
# High-ranking channel: POSITION (dot plot on a common scale)
# Low-ranking channel:  COLOUR (lighter = shorter petal)

# Map species to colours
species_palette = {'setosa': '#bbbbbb', 'versicolor': '#bbbbbb', 'virginica': '#ee6677'}
species_list = ['setosa', 'versicolor', 'virginica']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── High-ranking: position (strip / dot plot) ───────────────────────────
np.random.seed(42)
for i, sp in enumerate(species_list):
    sub = iris[iris['species'] == sp]
    jitter = np.random.uniform(-0.15, 0.15, len(sub))
    col = '#ee6677' if sp == 'virginica' else '#bbbbbb'
    ax1.scatter(sub['petal_length'], np.full(len(sub), i) + jitter,
                color=col, s=25, alpha=0.75, edgecolors='none')

ax1.set_yticks(range(3))
ax1.set_yticklabels(species_list)
ax1.set_xlabel('Petal length (cm)')
ax1.set_title('HIGH-RANKING CHANNEL: Position\nVirginica petals are distinctly longer —\n'
              'gap is readable directly from the axis scale',
              loc='left', fontsize=9)
for side in ('top', 'right'):
    ax1.spines[side].set_visible(False)

# ── Low-ranking: colour (lightness = petal_length) ──────────────────────
# Three columns of squares, each row = one species, lightness encodes petal_length
pmax = iris['petal_length'].max()
pmin = iris['petal_length'].min()

for i, sp in enumerate(species_list):
    sub = iris[iris['species'] == sp].reset_index(drop=True)
    for j, row in sub.iterrows():
        norm = (row['petal_length'] - pmin) / (pmax - pmin)  # 0=short, 1=long
        lightness = 0.95 - norm * 0.80
        rect = mpatches.Rectangle(
            (j * 0.08, i * 0.3 + 0.05), 0.07, 0.22,
            color=str(lightness)
        )
        ax2.add_patch(rect)

ax2.set_xlim(0, 4.2)
ax2.set_ylim(0, 1.1)
ax2.set_yticks([0.15, 0.45, 0.75])
ax2.set_yticklabels(species_list)
ax2.set_xticks([])
ax2.set_title('LOW-RANKING CHANNEL: Colour (lightness = petal_length)\n'
              'Virginica row is darker overall, but the magnitude of\n'
              'the difference is impossible to read off a shade',
              loc='left', fontsize=9)
for side in ('top', 'right', 'bottom'):
    ax2.spines[side].set_visible(False)

fig.suptitle('iris.csv — same petal_length data, two channels: position (high) vs colour (low)',
             x=0.01, ha='left')
plt.tight_layout()
plt.savefig('iris_channels.png', dpi=300, bbox_inches='tight')
plt.show()

print("""
Which to publish: POSITION.
A reader can read a numerical value off the x-axis and estimate the
gap between species to within ± 0.2 cm. With the colour encoding they
can only say 'darker = longer' and rank the species — they cannot
quantify the difference. The channel effectiveness ranking is not taste;
it is a statement about how accurately a reader can decode a number.
""")

## Limitations and why the published ranking still stands

**What this experiment can support:**  
With 6 ratios and 1 observer, I have 6 data points per channel and no statistical power. I cannot claim that position is *significantly* better than length, or compute a confidence interval. The direction of the ranking across all 5 channels (position best, colour worst) matches the published result, but one observer with 6 trials could easily reverse adjacent positions by chance.

**What this experiment cannot support:**  
I cannot claim that area is worse than angle for *all readers*, that the ranking is universal across ratio ranges, or that the effect sizes I measured match those in the literature.

**Why the published ranking is still worth following even when my small sample disagrees:**  
Cleveland & McGill (1984) ran the experiment on many observers across many trials and replicated it. The ranking reflects stable properties of the human visual system — position judgements use the most precise cortical machinery (V1 retinotopic map), while colour lightness comparisons are susceptible to simultaneous-contrast illusions. My personal experiment is a noisy demonstration of a well-established result, not an independent test of it. If my results disagreed, the most likely explanation is measurement noise, not that the ranking is wrong.